In [1]:
from functions import ping_openai, fetch_articles, read_prompt_from_file_only, load_function_schema, normalize_id, normalize_type, combine_paragraphs, format_nodes_for_prompt, get_schema
from model_dictionary import model_dictionary
from inputs import relationship_groups, groups_to_prompts, nodes_by_group_prompt, characteristic_node_types, required_node_types

import logging
import os
import certifi
from pymongo import MongoClient
from pymongo.server_api import ServerApi
from openai import OpenAI
from dotenv import load_dotenv
import json
from bson import json_util
import re
from datetime import datetime, timezone
load_dotenv()

# Get MongoDB credentials from environment variables
MONGO_URI = os.getenv("MONGO_URI")
DB_NAME = os.getenv("MONGO_DB_NAME")
ARTICLES_COLLECTION_NAME = os.getenv("MONGO_COLLECTION_NAME")

# Ensure all necessary environment variables are set
if not all([MONGO_URI, DB_NAME, ARTICLES_COLLECTION_NAME]):
    print("❌ Missing required environment variables. Check your .env file.")
    exit()

# Create MongoDB client with TLS verification
try:
    client = MongoClient(MONGO_URI, server_api=ServerApi('1'), tlsCAFile=certifi.where())
    db = client[DB_NAME]

    # Define collections
    articles_collection = db[ARTICLES_COLLECTION_NAME]  # Stores articles

    # Verify connection
    client.admin.command('ping')
    print(f"✅ Connected to MongoDB Atlas! Database: {DB_NAME}")

except Exception as e:
    print(f"❌ MongoDB Connection Error: {e}")
    exit()

openAI_api_key = os.getenv('OPENAI_API_KEY')
client = OpenAI(api_key=openAI_api_key)
    
ping_openai(client) 

✅ Connected to MongoDB Atlas! Database: tuone
✅ Successfully connected to OpenAI API!
Available Models: ['gpt-4o-realtime-preview-2024-12-17', 'gpt-4o-audio-preview-2024-12-17', 'dall-e-3', 'dall-e-2', 'gpt-4o-audio-preview-2024-10-01', 'gpt-4-turbo-preview', 'text-embedding-3-small', 'babbage-002', 'gpt-4', 'text-embedding-ada-002', 'chatgpt-4o-latest', 'gpt-4o-mini-audio-preview', 'gpt-4o-audio-preview', 'o1-preview-2024-09-12', 'gpt-4o-mini-realtime-preview', 'gpt-4o-mini-realtime-preview-2024-12-17', 'gpt-3.5-turbo-instruct-0914', 'gpt-4o-mini-search-preview', 'gpt-3.5-turbo-16k', 'gpt-4o-realtime-preview', 'davinci-002', 'gpt-3.5-turbo-1106', 'gpt-4o-search-preview', 'gpt-3.5-turbo-instruct', 'gpt-3.5-turbo', 'o3-mini-2025-01-31', 'gpt-4o-mini-search-preview-2025-03-11', 'gpt-4-0125-preview', 'gpt-4o-2024-11-20', 'gpt-4.1-nano-2025-04-14', 'gpt-4o-2024-05-13', 'o1-2024-12-17', 'o1', 'o1-preview', 'gpt-4-1106-preview', 'gpt-4-0613', 'o1-mini', 'gpt-4o-mini-tts', 'gpt-4o-transcribe'

In [2]:
def setup_logger(article_id, log_dir="logs"):
    os.makedirs(log_dir, exist_ok=True)
    logger = logging.getLogger(f"article_{article_id}")
    logger.setLevel(logging.DEBUG)

    if not logger.handlers:
        file_path = os.path.join(log_dir, f"{article_id}.log")
        handler = logging.FileHandler(file_path)
        formatter = logging.Formatter('%(asctime)s - %(levelname)s - %(message)s')
        handler.setFormatter(formatter)
        logger.addHandler(handler)

    return logger

In [3]:
def extract_nodes(article_id, text, PROMPT_PATH, FUNCTION_SCHEMA_PATH, model_name, logger):

    function_schema = load_function_schema(FUNCTION_SCHEMA_PATH)
    prompt = read_prompt_from_file_only(PROMPT_PATH)

    try:
        completion = client.chat.completions.create(
            model=model_name,
            messages=[
                {"role": "system", "content": prompt},
                {"role": "user", "content": f"Here is the article: {text}"}
            ],
            functions = [function_schema],
            function_call = {"name": "extract_clean_tech_entities"},
            temperature=0
        )

        # parse the returned function call
        function_args = completion.choices[0].message.function_call.arguments

        extracted_data = json.loads(function_args)
        logger.info("✅ Successful call to OpenAI. The following nodes are returned:")
        logger.info(json.dumps(extracted_data, indent=2, sort_keys=False))

        if "nodes" not in extracted_data:
            raise ValueError("❌ Expected 'nodes' key in LLM output but not found.")

        nodes_by_category = extracted_data["nodes"]

        # Flatten nodes while retaining their type
        formatted_nodes = []
        for node_type, node_categories in nodes_by_category.items():
            for node in node_categories:
                formatted_node = {
                    "article_id": article_id,
                    "id": normalize_id(node.get("id")),
                    "type": normalize_type(node.get("type")),
                    "name": node.get("name"),
                    "location": {
                        "city": node.get("location", {}).get("city", ""),
                        "country": node.get("location", {}).get("country", "")
                    } if node.get("location") else None,
                    "amount": node.get("amount"), #possibly to drop
                    "status": node.get("status"),
                }
                formatted_nodes.append(formatted_node)

        return formatted_nodes

    except json.JSONDecodeError as e:
        print(f"❌ JSON Parsing Error: {e}")
        return []
    except Exception as e:
        print(f"❌ Error extracting nodes: {e}")
        return [] 

In [4]:
def extract_node_characteristics(article_id, text, nodes, relationship_group, model_name, logger):

    PROMPT_PATH = groups_to_prompts[relationship_group]
    prompt = read_prompt_from_file_only(PROMPT_PATH)

    FUNCTION_SCHEMA_PATH = f"schemas/{relationship_group}.json"
    function_schema = load_function_schema(FUNCTION_SCHEMA_PATH)

    allowed_types = nodes_by_group_prompt[relationship_group]
    compact_nodes = format_nodes_for_prompt(nodes, allowed_types)
    
    try:
        completion = client.chat.completions.create(
            model=model_name,
            messages=[
                {"role": "system", "content": prompt},
                {
                "role": "user",
                 "content": f"""Here is the article text: {text}

                Here is the list of known entities:
                {compact_nodes}
                """
                }
            ],
            functions=[function_schema],
            function_call={"name": "extract_characteristics"},
            temperature=0
        )

        #parse the returned function call
        function_args = completion.choices[0].message.function_call.arguments
        extracted_data = json.loads(function_args)
        logger.info(f"✅ Succesful openAI call: returned the following capacity characteristics:")
        logger.info(json.dumps(extracted_data, indent=2, sort_keys=False))

        if "node_characteristics" not in extracted_data:
            raise ValueError("Expected 'node_characteristics' key in LLM output but not found.")

        return extracted_data["node_characteristics"]

    except json.JSONDecodeError as e:
        print(f"❌ JSON Parsing Error (relationships): {e}")
        return []
    except Exception as e:
        print(f"❌ Error extracting relationships: {e}")
        return [] 

In [5]:
def extract_relationships(article_id, text, nodes, relationship_group, model_name, logger, allowed_types=None):

    PROMPT_PATH = groups_to_prompts[relationship_group]
    prompt = read_prompt_from_file_only(PROMPT_PATH)

    FUNCTION_SCHEMA_PATH = f"schemas/{relationship_group}.json"
    function_schema = load_function_schema(FUNCTION_SCHEMA_PATH)
    
    allowed_types = nodes_by_group_prompt[relationship_group]
    compact_nodes = format_nodes_for_prompt(nodes, allowed_types)
    
    try:
        completion = client.chat.completions.create(
            model=model_name,
            messages=[
                {"role": "system", "content": prompt},
                {
                "role": "user",
                 "content": f"""Here is the article text: {text}

                Here is the list of known entities:
                {compact_nodes}

                Please extract only the specified relationship types."""
                }
            ],
            functions=[function_schema],
            function_call={"name": "extract_clean_tech_relationships"},
            temperature=0
        )

        #parse the returned function call
        function_args = completion.choices[0].message.function_call.arguments
        extracted_data = json.loads(function_args)
        logger.info(f"✅ Succesful openAI call: returned the following relationships:")
        logger.info(json.dumps(extracted_data, indent=2, sort_keys=False))

        if "relationships" not in extracted_data:
            raise ValueError("Expected 'relationships' key in LLM output but not found.")

        formatted_relationships = []
        for rel in extracted_data["relationships"]:
            raw_source = rel.get("source")
            raw_target = rel.get("target")
            norm_source = normalize_id(raw_source)
            norm_target = normalize_id(raw_target)

            #print(f"🔍 Normalizing IDs - Source: {raw_source} → {norm_source}, Target: {raw_target} → {norm_target}")

            formatted_relationships.append({
                "article_id": article_id,
                "id": f"{norm_source}_{rel.get('type')}_{norm_target}",
                "source": norm_source,
                "target": norm_target, # should this be changed to raw_target?
                "type": rel.get("type"),
                "group": relationship_group
            })

        return formatted_relationships

    except json.JSONDecodeError as e:
        print(f"❌ JSON Parsing Error (relationships): {e}")
        return []
    except Exception as e:
        print(f"❌ Error extracting relationships: {e}")
        return []

In [16]:
def process_articles(articles_to_process, PROMPT_PATH, FUNCTION_SCHEMA_PATH, model_dictionary):

    for article in articles_to_process:
        articleID = str(article["_id"])  #convert ObjectId to string

        logger_articleID = article['meta']['ID']
        logger = setup_logger(logger_articleID)
        logger.info(f"📌 Processing Article ID: {logger_articleID}")
        print(f"📌 Processing Article ID: {logger_articleID} - {article["title"]}")

        val = article.get("validation")
        if val is True:
            print("⏭️  Skipping – article is validated")
            continue
        elif isinstance(val, (int, float)):           # timestamp
            processed_on = datetime.fromtimestamp(val, tz=timezone.utc) \
                                    .strftime("%Y‑%m‑%d %H:%M UTC")
            print(f"⏭️  Skipping – article was validated on {processed_on}")
            continue

        text = combine_paragraphs(article)
        if not text:
            logger.info(f"⚠️ No valid text found for Article ID: {articleID}. Skipping.")
            continue

        logger.info(f"📌 Processing Article ID: {articleID}")

        try:
            # extract entities
            formatted_nodes = extract_nodes(articleID, text, PROMPT_PATH, FUNCTION_SCHEMA_PATH, model_dictionary["nodes"], logger)
            logger.info(f"Formatted nodes used for all prompts hereafter: {formatted_nodes}")

            # attach characteristics to entities (capacities and investments)
            for relationship_group, config in characteristic_node_types.items():
                model_name = model_dictionary[relationship_group] # select fine-tuned model
                id_key = config["id_key"]
                type_match = config["type_match"]

                # only continue if there are nodes of this type in the article
                has_relevant_nodes = any(node.get("type") == type_match for node in formatted_nodes)
                if not has_relevant_nodes:
                    logger.info(f"⏭️ Skipping {relationship_group} – no nodes of type '{type_match}' found.")
                    continue

                logger.info(f"🔍 Extracting characteristics for node group: {relationship_group}")
                node_characteristics = extract_node_characteristics(articleID, text, formatted_nodes, relationship_group, model_name, logger)  

                # attach 'status' and 'type' to matching capacity nodes (flat list structure)
                for char in node_characteristics:
                    node_id = char.get(id_key)
                    logger.info(f"Node ID is {node_id}")
                    found_match = False

                    for node in formatted_nodes:
                        if node.get("id") == node_id and node.get("type") == type_match:
                            logger.info(f"✅ Match found for {id_key} '{node_id}' in {type_match} nodes")
                            if "status" in char:
                                node["status"] = char["status"]
                                logger.info(f"  ➕ Set status: {char['status']}")
                            if "phase" in char:
                                node["phase"] = char["phase"]
                                logger.info(f"  ➕ Set phase: {char['phase']}")
                            found_match = True

                    if not found_match:
                        logger.info(f"⚠️ No match found for {id_key} '{node_id}' in formatted_nodes")

            # extract relationships between entities
            all_relationships = []
            for relationship_group,model_name in model_dictionary.items():
                if relationship_group in ["nodes", "capacities", "investments"]: #skip nodes, capacities and investments which have logic elsewhere
                    continue

                def has_required_nodes(formatted_nodes, required_types):
                    existing_types = {node['type'] for node in formatted_nodes}
                    return any(req_type in existing_types for req_type in required_types)
                
                required_types = required_node_types.get(relationship_group, [])
                if not has_required_nodes(formatted_nodes, required_types):
                    logger.info(f"🛑 Skipping {relationship_group}: required nodes {required_types} not present.")
                    continue

                logger.info(f"- - - Querying openai for relationships: {relationship_group}, using model: {model_name}")

                allowed_types = relationship_groups[relationship_group]
                logger.info(f"Node types included in the query: {allowed_types}")

                relationships = extract_relationships(articleID, text, formatted_nodes, relationship_group, model_name, logger, allowed_types)
                all_relationships.extend(relationships)

            # update entries in mongodb with clean entities and relationships
            update_result = articles_collection.update_one(
                {"_id": article["_id"]},  #match by article id
                {"$set": {
                    "nodes": formatted_nodes or [],
                    "relationships": all_relationships or []
                    }})

            if update_result.modified_count > 0:
                logger.info(f"✅ Updated Article ID: {articleID} with {len(formatted_nodes)} nodes and combined text.")
            else:
                logger.info(f"⚠️ No updates made for Article ID: {articleID}.")

        except Exception as e:
            print(f"❌ Error processing Article ID {articleID}: {e}")
            

In [21]:
n_articles = 0
offset_articles = 5

articles_to_process = list(
    articles_collection.find(
        {"meta.ID": {"$regex": "^13"}},  
        {"_id": 1, "meta": 1,
        "paragraphs": 1, "validation": 1, "title": 1}       
    )
    .sort("_id", -1)  # Sort by MongoDB ObjectId (descending)
    .skip(offset_articles)    # Skip first `offset` articles
    .limit(n_articles)    # Limit the number of articles
)

PROMPT_PATH = "prompts/entities-only.txt"
FUNCTION_SCHEMA_PATH = "schemas/entities.json"

process_articles(articles_to_process, PROMPT_PATH, FUNCTION_SCHEMA_PATH, model_dictionary)

📌 Processing Article ID: 1304554 - US Government to Fund Research to Improve Offshore Wind and Marine Energy Deployments
📌 Processing Article ID: 1304536 - Lamprell Secures Norfolk Vanguard Transition Piece Job
📌 Processing Article ID: 1304533 - Siemens Gamesa and Ørsted Firm Up Changhua Order
⏭️  Skipping – article was validated on 1970‑01‑01 00:00 UTC
📌 Processing Article ID: 1304530 - Waves Group Joins Hollandse Kust Zuid Team
📌 Processing Article ID: 1304525 - Leading Light Wind to Invest in Staten Island as Offshore Wind Hub
📌 Processing Article ID: 1304510 - SeAH to Deliver Monopiles for Vattenfall’s 2.8 GW Norfolk Vanguard Offshore Wind Project
📌 Processing Article ID: 1304504 - K2 Management Enters South Korean Offshore Wind Deal
📌 Processing Article ID: 1304496 - EEW SPC Secures 176-Monopile Order for Coastal Virginia Offshore Wind
📌 Processing Article ID: 1304492 - Ørsted Awards Secondary Steel Contract for New Jersey’s First Offshore Wind Farm
📌 Processing Article ID: 130449

KeyboardInterrupt: 

In [47]:
from bson import ObjectId

article_id = "67f52cfa981040986eab7b38"

# Then query it using ObjectId
article = articles_collection.find_one({"_id": ObjectId(article_id)})
article['meta']['ID']

'2708506'

In [19]:

target_id = ObjectId("67f52cfa981040986eab7b38")

for idx, article in enumerate(articles_to_process):
    if article.get("_id") == target_id:
        print(f"✅ Found at position {idx}")
        break
else:
    print("❌ Article not found")

✅ Found at position 52


In [74]:
articles_collection.find_one({"title": "VW may save its German factories after all"})

{'_id': ObjectId('67f52d07981040986eab7b6c'),
 'title': 'VW may save its German factories after all',
 'paragraphs': [{'p1': 'According to insiders, a possible solution is emerging in the wage dispute at Volkswagen in order to retain all VW brand plants in Germany. Nevertheless, it would be a turning point for the Zwickau electric car plant - something similar is also emerging for the iconic Golf.',
   'p2': 'As reported by theBloombergnews agency, the management is prepared to keep the plants running and reintroduce job security agreements until 2030 in return for employees waiving their bonus payments. Several informants are said to have stated this, but are not described in detail.',
   'p3': 'The fifth round of negotiations between Volkswagen and IG Metall was scheduled for this week. Both sides had emphasised in advance that they were aiming to reach an agreement before Christmas. However, despite several marathon negotiations, there was no officially confirmed breakthrough in the

In [7]:
offset = 3
limit = 1

articles_to_process = list(
    articles_collection.find(
        {"meta.ID": {"$regex": "^23"}},  # <-- Add this filter
        {"_id": 1, "paragraphs": 1, "validation": 1, "meta": 1, "title": 1,
         "nodes":1, "relationships":1}       # Keep projection as is
    )
    .sort("_id", -1)  # Sort by MongoDB ObjectId (descending)
    .skip(offset)    # Skip first `offset` articles
    .limit(limit)    # Limit the number of articles
)

articles_to_process

[{'_id': ObjectId('67fa9a3ceaa4fb525afef9b1'),
  'title': 'TMEIC to Open 9 GW Utility-Scale Solar Inverter Factory in Texas',
  'paragraphs': [{'p1': 'TMEIC has said that it will relocate its US headquarters to Houston, Texas, in March 2025. The move will coincide with the establishment of a new manufacturing facility in Brookshire, Texas.',
    'p2': 'TMEIC Corporation Americas will manufacture utility-scale solar inverters at the facility. It said the site is large enough to scale up to an annual production capacity of 9 GW of inverters and will scale based on demand.',
    'p3': 'The company has more than 50 GW of its products installed and operational throughout the world, with 28 GW installed in North America alone.',
    'p4': 'It is moving its headquarters to Texas from Roanoke, Virginia, for the site expansion. However, it will maintain its office in Roanoke, Virginia, with a focus on designing, developing, and engineering advanced automation systems, large AC motors, and varia

Validation tracking

In [27]:
validated_count = articles_collection.count_documents({"validation": True})
print(f"Number of validated articles: {validated_count}")

# If you want to specifically count only electric vehicle articles that are validated:
ev_validated_count = articles_collection.count_documents({
    "validation": True,
    "meta.ID": {"$regex": "^27"}  # Same filter as in your process_relationships function
})
print(f"Number of validated Electrive Automobile articles: {ev_validated_count}")

# If you want to specifically count only electric vehicle articles that are validated:
ev_validated_count = articles_collection.count_documents({
    "validation": True,
    "meta.ID": {"$regex": "^33"}  # Same filter as in your process_relationships function
})
print(f"Number of validated World Energy Vehicle articles: {ev_validated_count}")

Number of validated articles: 132
Number of validated Electrive Automobile articles: 41
Number of validated World Energy Vehicle articles: 68
